# 1. LEARNABLE β — AeroSync-Mamba v19

v19 starts from v18. It asks: **if the model sets its own lag-bias strength, does the lag bias help, and where does β end up?**

**What came before (same 5 seeds, macro-F1, β = 1 minus β = 0):**
- **v17, fixed β = 1.0, un-stabilized:** mean paired difference **+0.0008**, std of the differences **0.0020**, β = 1 won on **3/5** seeds. Not established.
- **v18, fixed β = 1.0 with LayerNorm + zero-init + 5-epoch warm-up:** mean paired difference **−0.0094**, std **0.0136**, β = 1 won on **1/5** seeds (β = 0 was better on 4 of 5). The paired t-test gave p = 0.198. So the direction is fairly consistent, but the size of the harm is not statistically established.

**What changes in v19:**
- **Kept from v18:** the LayerNorm over each window's (T_m, T_n) lag-bias grid (no parameters) and the zero-initialized final layer of g. The LayerNorm code is copied unchanged and checked in Section 4.
- **Removed from v18:** the fixed β warm-up schedule.
- **New:** β = softplus(raw_β). This is **one learnable scalar shared by both LACA directions**: not per head, not per direction. raw_β = log(expm1(0.005)) ≈ −5.296, so **β(0) ≈ 0.0050** and **dβ/draw_β = sigmoid(raw_β) ≈ 0.0050**. Section 4 explains this choice and why raw_β gets its own learning rate.

**Design.** 5 new runs, one per seed (42, 123, 2024, 7, 99).
- **The β = 0.0 control is not retrained.** v18's five β = 0.0 rows are reused directly. That condition had the same LayerNorm and zero-init g, with β held at exactly 0, so the warm-up had no effect on it. It already reproduced v17's β = 0.0 arm exactly.
- **Pairing is checked per seed.** Apart from raw_β, each new run's initial weights and RNG state must match the v18 control run for that seed.
- **Section 7** reports the same paired statistics as v17 and v18, a comparison of all four β approaches, and **where β converged**, analyzed as carefully as the accuracy numbers.

| Decision | Value | Evidence status |
|---|---|---|
| Timestamps into HTT / LACA | Real per-window audio and IMU timestamps (not a synthetic `linspace`) | CORRECTNESS FIX (earlier versions passed an identical synthetic grid for every window) |
| Fusion backbone | Real `mamba_ssm.Mamba` selective-scan SSM; pure-PyTorch fallback only when no CUDA GPU is available (printed unmissably) | CORRECTNESS FIX (the earlier gated-conv block was not an SSM) |
| Audio bin selection | Strongest-N FFT bins by mean magnitude, computed once from **training** audio only, then frozen and reused for val/test | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): strongest-64 0.8736 vs first-64 0.8494, +0.0242 (bin-selection study) |
| Frequency budget N | 64 bins | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v6: N=64 0.8966 vs N=48 0.8863, **+0.0103**, which is below 0.0160. The other budgets clear it (N=32 +0.0249, N=128 +0.0166) except N=96 (+0.0142). |
| Audio compression | `sqrt(|rFFT|)` | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 sqrt 0.8966 vs log_db 0.8696 (+0.0270) and log1p 0.8649 (+0.0317) |
| Audio windowing | None; Hann windowing rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 no window 0.8649 vs Hann 0.8290 (+0.0359) |
| Audio normalization | None; z-score normalization rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 none 0.8966 vs z-score 0.8684 (+0.0282) |
| Audio time positions Ta | 64 per 2.56 s window | PROVISIONAL: v7 Ta=128 0.8967 vs Ta=64 0.8966 (−0.0001, a tie). The cheaper Ta=64 is kept. |
| Sensor input | Raw 6 channels (accel x/y/z, gyro x/y/z), Ts=128; derived magnitude features rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v8 raw6 0.8966 vs best derived set 0.8740 (+0.0226) |
| LACA temporal-bias scale β | **v19: learnable**, β = softplus(raw_β), one global scalar, β(0) = 0.005 | **UNDER TEST IN THIS NOTEBOOK.** Fixed β = 1.0 is NOT ESTABLISHED: v17 (un-stabilized) mean diff +0.0008, std 0.0020, 3/5 wins; v18 (stabilized + warm-up) mean diff −0.0094, std 0.0136, 1/5 wins, p = 0.198. |
| LACA lag-bias form g(Δτ) | mlp `Linear(1,32) → Tanh → Linear(32,1)`, 97 params. **v18:** final Linear zero-initialized, output LayerNorm over each window's (T_m, T_n) grid (0 params) | PROVISIONAL: v10 mlp vs rbf +0.0029, vs bucketed +0.0030 (below 0.0160). The v18 stabilization is under test. |
| LACA heads | 4 | PROVISIONAL: v11 h=1 was +0.0008 (one test window) and h=4 was kept. h=2/8/16 were 0.006–0.012 worse (below 0.0160). |
| Mamba fusion | 2 blocks, d_state=16, d_conv=2 (expand 2) | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v12: 2 vs 4 blocks +0.0023, d_state 16 vs 8 +0.0172, d_conv 2 vs 4 +0.0059. |
| λ_align (InfoNCE) | 0.1 | PROVISIONAL: v13 removing L_align cost −0.0088, which is below 0.0160. λ=0.05 was −0.0271 and λ=0.2 was −0.0070. |
| λ_cons (KL consistency) | 0.1 | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v13 grid: λ=0.2 +0.0008, λ=0.05 −0.0232. Removing L_cons cost −0.0165. |
| Ordinal-severity loss | Not used (rejected) | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v14 best λ_ord>0 0.8859 vs none 0.9048 (−0.0189); adjacent-class errors rose 94 → 113 |
| Classification loss | Cross-entropy weighted by inverse **training-set** class frequency | CORRECTNESS FIX (class-imbalance handling) |
| Seeding | Python `random`, numpy, torch CPU + all CUDA devices, cuDNN deterministic; reset per seed at run start and right before model construction. The mission split has its own fixed seed (42), so every seed sees the same train/val/test windows. | CORRECTNESS FIX (reproducibility) |

## 2. Configuration constants & environment
Every fixed value lives in the first cell below. The second installs `mamba_ssm` and reports which fusion backbone is active.

In [1]:
import copy
import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
import urllib.request
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==============================================================================
# FIXED CONFIGURATION — every finalized choice (see the summary table above)
# ==============================================================================
# --- Learnable-beta runs: one per seed (the beta=0.0 control is REUSED from v18, not retrained) ---
SEEDS = [42, 123, 2024, 7, 99]           # same 5 seeds as every prior beta experiment, in this order
SPLIT_SEED = 42                          # mission-level train/val/test split (independent of the training seed)

# --- Dataset / split ---
DATASET_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
WORK_DIR = Path("/kaggle/working")
DATASET_BASE = WORK_DIR / "tii_uav_dataset"
DATASET_ROOT = DATASET_BASE / "Dataset"
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
N_CLASSES = 5                            # propeller-fault severity 0..4

# --- Windowing ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
MIN_SENSOR_SAMPLES_PER_WINDOW = 10

# --- Audio: strongest-64 bins (train-only), sqrt(|rFFT|), no window, no normalization ---
N_AUDIO_BINS = 64                        # frequency axis
TA_AUDIO = 64                            # audio time positions per window

# --- Sensor: raw 6 channels (accel x/y/z, gyro x/y/z), no derived features ---
N_SENSOR_CHANNELS = 6
TS_SENSOR = 128                          # sensor time positions per window
SENSOR_STD_EPS = 1e-6

# --- Model ---
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LAG_BIAS_HIDDEN = 32                     # mlp g(Δτ): Linear(1,32) -> Tanh -> Linear(32,1) = 97 params
LAG_BIAS_PARAMS = 97                     # the v18 LayerNorm has no parameters -> still 97
LAG_BIAS_LN_EPS = 1e-5                   # v18 LayerNorm over each window's (T_m, T_n) lag-bias grid, no affine (kept)
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496          # 2 x 437,248 (real mamba_ssm)
DROPOUT = 0.15

# --- Loss: L_cls (class-weighted CE) + 0.1 * L_align (InfoNCE) + 0.1 * L_cons (KL) ---
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- Optimization ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- v19: learnable LACA beta = softplus(raw_beta) — ONE global scalar shared by both LACA directions ---
BETA_INIT = 0.005                                 # beta at initialization (requested range 0.001-0.01)
RAW_BETA_INIT = math.log(math.expm1(BETA_INIT))   # inverse softplus -> -5.2958; d(beta)/d(raw) = sigmoid(raw) ~ 0.0050
MIN_INIT_GRAD = 1e-4                              # d(beta)/d(raw_beta) at init must exceed this (checked for every seed)
BETA_LR = 1e-2                                    # AdamW lr for raw_beta ONLY (every other parameter keeps LEARNING_RATE)
BETA_WEIGHT_DECAY = 0.0                           # no decay on raw_beta: decay pulls raw -> 0, i.e. beta -> softplus(0) = 0.693
BETA_CHECKPOINT_EPOCHS = [1, 5, 10, EPOCHS]       # beta trajectory logged at these epochs (plus every epoch in the output)
# Where-did-beta-converge thresholds. After the LayerNorm the lag term has unit std per window, and its max |z| was
# ~1.2-2.5 in v18 at beta=1. beta <= 0.05 -> max contribution <~0.1 logit (negligible); beta >= 0.2 -> ~0.25-0.5,
# comparable to v17's un-stabilized contributions (0.23-0.68), i.e. a real effect on attention.
BETA_NEAR_ZERO = 0.05
BETA_NONTRIVIAL = 0.2
BETA_CONSISTENT_MAXMIN = 3.0                      # "consistent" nontrivial beta: max/min across seeds <= 3

# --- Outputs ---
CHECKPOINT_DIR = WORK_DIR / "v19_checkpoints"
PROBS_DIR = WORK_DIR / "v19_test_probs"
RESULTS_CSV = WORK_DIR / "beta_learnable_paired_results.csv"
for _d in (CHECKPOINT_DIR, PROBS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Prior beta experiments on the same 5 seeds (copied from Artifacts/Version 17 and Version 18 CSVs) ---
# CONTROL, REUSED — NOT RETRAINED: v18's beta=0.0 runs. Same architecture as this notebook's baseline needs (v18
# LayerNorm + zero-init g, beta hard-fixed at exactly 0 every epoch -> warm-up was a no-op; lag term asserted exactly 0
# at every epoch). These rows also reproduced v17's beta=0.0 rows exactly. init_sha1 / rng_sha1 are used to verify
# that each new run starts from the same initial weights and RNG stream as its control (raw_beta excluded).
V18_BETA0_CONTROL = {
    42: {"accuracy": 0.9088495575221239, "macro_f1": 0.9057330017972403, "balanced_accuracy": 0.9030033658152373, "macro_auroc": 0.9891867537737307,
          "init_sha1": "5850841e699f", "rng_sha1": "5641b401720d"},
    123: {"accuracy": 0.8876106194690265, "macro_f1": 0.8826825575079713, "balanced_accuracy": 0.8790467961967972, "macro_auroc": 0.9861866752615566,
          "init_sha1": "74be0883c62b", "rng_sha1": "0a4b3be44b18"},
    2024: {"accuracy": 0.8805309734513275, "macro_f1": 0.8763643000413779, "balanced_accuracy": 0.8715485268344848, "macro_auroc": 0.9866958753709998,
          "init_sha1": "92852955eef1", "rng_sha1": "619c0e0a6b4d"},
    7: {"accuracy": 0.9061946902654867, "macro_f1": 0.9019350016739314, "balanced_accuracy": 0.9008047135870789, "macro_auroc": 0.9897445660698647,
          "init_sha1": "7f5d0cd2e7cb", "rng_sha1": "9fdcf777d285"},
    99: {"accuracy": 0.8690265486725663, "macro_f1": 0.8638026478536822, "balanced_accuracy": 0.8598248510633313, "macro_auroc": 0.9829841151452052,
          "init_sha1": "e40c78169f57", "rng_sha1": "81dcd4638a88"},
}
V17_BETA1_MACRO_F1 = {42: 0.9047540152267212, 123: 0.8818976370764308, 2024: 0.8783309161428443, 7: 0.9020733412115728, 99: 0.8674825744048924}   # fixed beta=1.0, un-stabilized
V18_BETA1_MACRO_F1 = {42: 0.876757425721965, 123: 0.8812179318576554, 2024: 0.8652112392086446, 7: 0.8891867251928065, 99: 0.8711810957395529}   # fixed beta=1.0, stabilized + warm-up
V17_BETA1_LAG_CONTRIB = {42: 0.42617592215538025, 123: 0.23168042302131653, 2024: 0.6771183609962463, 7: 0.2886081039905548, 99: 0.3633767366409302}   # max|beta*g|, best ckpt
V18_BETA1_LAG_CONTRIB = {42: 1.8926622867584229, 123: 1.9956361055374146, 2024: 2.4582314491271973, 7: 1.2155512571334839, 99: 2.11137056350708}   # max|beta*LN(g)|, best ckpt
V16_BASELINE_SEED42 = {"accuracy": 0.9080, "macro_f1": 0.9048, "balanced_accuracy": 0.9022, "macro_auroc": 0.9894}
SAME_SEED_TOLERANCE = 0.0005             # same-seed reproducibility (code correctness), not a finding threshold
SEED_TO_SEED_STD_F1 = 0.0160             # v15/v16 measured seed-to-seed std of test macro-F1
PAIRED_ALPHA = 0.05
REFERENCE_SEED42_RNG_PREFIX = "5641b401720d"   # torch CPU RNG right after seed-42 model construction (v10-v15)
REFERENCE_DATA = {
    "bin_indices": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                    258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                    509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                    761, 762, 763, 764, 765, 766],
    "sensor_fingerprint": {
        "train": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
        "val":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
        "test":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
    },
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


def set_all_seeds(seed: int) -> None:
    """Seed python `random`, numpy, torch CPU and every CUDA device; deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print(f"LEARNABLE beta: {len(SEEDS)} runs (seeds {SEEDS}) vs the REUSED v18 beta=0.0 control (not retrained) | "
      f"results -> {RESULTS_CSV}")
print(f"beta = softplus(raw_beta), one global scalar | raw_beta init {RAW_BETA_INIT:.4f} -> beta(0) {BETA_INIT} | raw_beta "
      f"lr {BETA_LR}, weight decay {BETA_WEIGHT_DECAY} | kept from v18: LayerNorm over each window's (T_m, T_n) grid "
      f"(eps {LAG_BIAS_LN_EPS}, no affine) + zero-initialized g final layer | removed: beta warm-up")
print(f"Fixed configuration: audio strongest-{N_AUDIO_BINS}, Ta={TA_AUDIO}, sqrt, no window, no normalization "
      f"| sensor raw {N_SENSOR_CHANNELS} ch, Ts={TS_SENSOR} | LACA beta learnable, {LACA_NUM_HEADS} heads, mlp g(Δτ) "
      f"{LAG_BIAS_PARAMS} params | Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) | "
      f"loss CE + {LAMBDA_ALIGN} InfoNCE + {LAMBDA_CONS} KL | {EPOCHS} epochs, batch {BATCH_SIZE}, AdamW {LEARNING_RATE}")

Device: cuda (Tesla T4)
LEARNABLE beta: 5 runs (seeds [42, 123, 2024, 7, 99]) vs the REUSED v18 beta=0.0 control (not retrained) | results -> /kaggle/working/beta_learnable_paired_results.csv
beta = softplus(raw_beta), one global scalar | raw_beta init -5.2958 -> beta(0) 0.005 | raw_beta lr 0.01, weight decay 0.0 | kept from v18: LayerNorm over each window's (T_m, T_n) grid (eps 1e-05, no affine) + zero-initialized g final layer | removed: beta warm-up
Fixed configuration: audio strongest-64, Ta=64, sqrt, no window, no normalization | sensor raw 6 ch, Ts=128 | LACA beta learnable, 4 heads, mlp g(Δτ) 97 params | Mamba 2 x (d_state 16, d_conv 2) | loss CE + 0.1 InfoNCE + 0.1 KL | 20 epochs, batch 32, AdamW 0.0001


In [2]:
# ==============================================================================
# mamba_ssm installation + fusion-backbone resolution.
# Order: an already-installed package, else a prebuilt wheel matched to this
# Python / torch / CUDA / C++ ABI (the reliable path on Kaggle), else a
# time-capped source build. If none succeeds, or there is no CUDA GPU, the
# pure-PyTorch fallback block is used and announced below.
# ==============================================================================
PIP_TIMEOUT_S = 240


def _env_tags():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    try:
        cxx11_abi = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except AttributeError:
        cxx11_abi = False
    return py_tag, torch_mm, torch.version.cuda, cxx11_abi


def _prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, cxx11_abi):
    """Best-matching .whl asset in `repo`'s GitHub releases (exact CUDA version first, then CUDA major)."""
    req = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases",
                                 headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.load(resp)
    except Exception as exc:
        print(f"    could not query github.com/{repo}/releases: {exc!r}")
        return None
    abi = f"cxx11abi{'TRUE' if cxx11_abi else 'FALSE'}"
    cuda_exact = f"cu{cuda_ver.replace('.', '')}"
    cuda_major = f"cu{cuda_ver.split('.')[0]}"
    exact, major_only = [], []
    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not (name.endswith(".whl") and py_tag in name and abi in name and f"torch{torch_mm}" in name):
                continue
            if cuda_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major in name:
                major_only.append(asset["browser_download_url"])
    chosen = exact or major_only
    return chosen[0] if chosen else None


def _pip_install(args):
    """pip with a hard timeout and captured output, so a stuck build can never hang the kernel."""
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                              timeout=PIP_TIMEOUT_S, capture_output=True, text=True, check=False)
    except subprocess.TimeoutExpired:
        print(f"    pip install {args} exceeded {PIP_TIMEOUT_S}s — killed")
        return False
    if proc.returncode != 0:
        print(f"    pip install {args} failed (exit {proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")
    return proc.returncode == 0


def install_mamba_ssm():
    if importlib.util.find_spec("mamba_ssm") is not None:
        return "already installed"
    tags = _env_tags()
    print(f"Installing mamba_ssm + causal-conv1d (python={tags[0]}, torch={tags[1]}, cuda={tags[2]}, cxx11abi={tags[3]})")
    urls = [_prebuilt_wheel_url(repo, *tags) for repo in ("Dao-AILab/causal-conv1d", "state-spaces/mamba")]
    if all(urls) and all(_pip_install([u, "--no-deps"]) for u in urls):
        return "prebuilt wheel"
    print("  no complete prebuilt-wheel match — trying a time-capped source build")
    if _pip_install(["causal-conv1d>=1.2.0", "--no-build-isolation"]) and _pip_install(["mamba-ssm", "--no-build-isolation"]):
        return "source build"
    return "failed"


MAMBA_CLASS = None
if torch.cuda.is_available():
    print(f"mamba_ssm install: {install_mamba_ssm()}")
    try:
        from mamba_ssm import Mamba as MAMBA_CLASS
    except ImportError as exc:
        print(f"mamba_ssm import failed: {exc!r}")

if MAMBA_CLASS is not None:
    FUSION_BACKBONE = "mamba_ssm.Mamba (real selective-scan SSM, CUDA)"
    print(f"✅ FUSION BACKBONE: {FUSION_BACKBONE}")
else:
    FUSION_BACKBONE = "FALLBACK pure-PyTorch gated-conv block (NOT a state-space model)"
    print("🚨" * 45)
    print(f"🚨  FUSION BACKBONE: {FUSION_BACKBONE}")
    print("🚨  " + ("no CUDA GPU detected" if not torch.cuda.is_available() else "mamba_ssm could not be installed/imported")
          + " — results will NOT match the reference numbers, which were produced with real mamba_ssm.")
    print("🚨" * 45)

Installing mamba_ssm + causal-conv1d (python=cp312, torch=2.10, cuda=12.8, cxx11abi=True)
    pip install ['https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl', '--no-deps'] failed (exit 1): ERROR: causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl is not a supported wheel on this platform.

  no complete prebuilt-wheel match — trying a time-capped source build
mamba_ssm install: source build
✅ FUSION BACKBONE: mamba_ssm.Mamba (real selective-scan SSM, CUDA)


## 3. Data loading & preprocessing
Mission-level split, strongest-64 bin selection from training audio only, 2.56 s windows with real timestamps, and sensor standardization with training-split statistics. Leakage and timestamp checks are included.

In [3]:
# ==============================================================================
# DATA — TII UAV Realistic Fault Dataset: parse missions -> mission-level split
# -> strongest-64 bins from TRAINING audio only (frozen) -> 2.56 s windows with
# real per-window timestamps -> datasets (sensor standardized with TRAINING stats).
# ==============================================================================
if not DATASET_ROOT.exists():
    print(f"Cloning {DATASET_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(DATASET_BASE)], check=True)


def parse_mission(mission_dir: Path):
    """Real 6-axis IMU (seconds from mission start) + raw audio chunks with per-chunk timestamps.
    Audio chunk duration = the mission's IMU-measured duration / number of chunks (both come from the same
    continuous recording; the audio log carries no per-chunk timestamp)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro = [], [], []
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks = []
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    chunk_duration_s = (sensor_times[-1] - sensor_times[0]) / len(audio_chunks)
    audio_times = np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks}


def split_missions(dataset_root: Path):
    """Mission-level stratified split (70 / 15 / 15 per class) with its own fixed seed."""
    class_missions = defaultdict(list)
    for class_dir in sorted(dataset_root.iterdir()):
        if not class_dir.is_dir() or class_dir.name.startswith("."):
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        for mission_dir in sorted(class_dir.iterdir()):
            if mission_dir.is_dir() and not mission_dir.name.startswith("."):
                class_missions[label].append(mission_dir)

    random.seed(SPLIT_SEED)
    splits = {"train": {}, "val": {}, "test": {}}
    for label in sorted(class_missions):
        missions = class_missions[label]
        random.shuffle(missions)
        n_train = max(1, int(len(missions) * TRAIN_FRACTION))
        n_val = max(1, int(len(missions) * VAL_FRACTION))
        splits["train"][label] = missions[:n_train]
        splits["val"][label] = missions[n_train:n_train + n_val]
        splits["test"][label] = missions[n_train + n_val:]
        print(f"Class {label} ({len(missions)} flights) -> train {n_train}, val {len(splits['val'][label])}, "
              f"test {len(splits['test'][label])}")
    return splits


def select_strongest_bins(train_missions, n_bins):
    """Indices of the n_bins rFFT magnitude bins with the highest mean magnitude over TRAINING audio only
    (unsupervised energy, no labels). Returned sorted; frozen and reused for val/test."""
    magnitude_sum, n_chunks, fft_len = None, 0, None
    for mission in train_missions:
        mag = np.abs(np.fft.rfft(mission["audio_chunks"], axis=1))
        fft_len = mag.shape[1] if fft_len is None else min(fft_len, mag.shape[1])
        mag = mag[:, :fft_len]
        if magnitude_sum is not None and len(magnitude_sum) > fft_len:
            magnitude_sum = magnitude_sum[:fft_len]
        magnitude_sum = mag.sum(axis=0) if magnitude_sum is None else magnitude_sum + mag.sum(axis=0)
        n_chunks += mag.shape[0]
    mean_magnitude = magnitude_sum / n_chunks
    return np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)


def audio_features(audio_chunks, bin_indices):
    """raw chunk -> rFFT (no window) -> |.| -> frozen strongest bins -> sqrt (no normalization)."""
    mag = np.abs(np.fft.rfft(audio_chunks, axis=1))
    valid = bin_indices[bin_indices < mag.shape[1]]
    feats = np.zeros((mag.shape[0], len(bin_indices)), dtype=np.float32)
    feats[:, :len(valid)] = np.sqrt(mag[:, valid]).astype(np.float32)
    return feats


def slice_windows(mission, label, bin_indices):
    """2.56 s windows (1.28 s step). Sensor: TS_SENSOR real IMU samples chosen evenly by index inside the window;
    audio: TA_AUDIO real chunk timestamps chosen the same way (a window with fewer than 2 audio chunks — the
    recording edge — falls back to an even grid). tau_a / tau_s = those REAL timestamps relative to window start."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


class UAVWindowDataset(Dataset):
    """Windows -> tensors. Sensor channels are standardized with mean/std that must come from the TRAINING
    split (computed here only when none are passed, i.e. for the training set). tau is normalized by the
    window length."""

    def __init__(self, windows, sensor_mean=None, sensor_std=None):
        self.windows = windows
        if sensor_mean is None or sensor_std is None:
            all_s = np.concatenate([w["x_s"] for w in windows], axis=0)
            sensor_mean = all_s.mean(axis=0, keepdims=True)
            sensor_std = all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS
        self.sensor_mean, self.sensor_std = sensor_mean, sensor_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        window_sec = float(w["window_sec"])
        return {
            "x_a": torch.tensor(w["x_a"], dtype=torch.float32),
            "x_s": torch.tensor((w["x_s"] - self.sensor_mean) / self.sensor_std, dtype=torch.float32),
            "tau_a": torch.tensor(w["tau_a"], dtype=torch.float32) / window_sec,
            "tau_s": torch.tensor(w["tau_s"], dtype=torch.float32) / window_sec,
            "label": torch.tensor(w["label"], dtype=torch.long),
        }


def split_fingerprint(windows):
    """(n windows, sum of raw sensor values, sum of sensor timestamps, class counts) — deterministic data check."""
    x_s = np.stack([w["x_s"] for w in windows]).astype(np.float64)
    tau_s = np.stack([w["tau_s"] for w in windows]).astype(np.float64)
    counts = tuple(np.bincount([w["label"] for w in windows], minlength=N_CLASSES).tolist())
    return (len(windows), round(float(x_s.sum()), 3), round(float(tau_s.sum()), 3), counts)


# --- 1. split + parse ---
mission_splits = split_missions(DATASET_ROOT)
split_dirs = {s: [m for label in sorted(mission_splits[s]) for m in mission_splits[s][label]] for s in mission_splits}
assert not set(split_dirs["train"]) & (set(split_dirs["val"]) | set(split_dirs["test"])), "LEAKAGE: mission in two splits"
parsed_missions = {m: parse_mission(m) for s in split_dirs for m in tqdm(split_dirs[s], desc=f"parse {s}", leave=False)}

# --- 2. strongest-64 bins from TRAINING audio only ---
bin_source_missions = [m for m in split_dirs["train"] if parsed_missions[m] is not None]
assert set(bin_source_missions) <= set(split_dirs["train"])
SELECTED_BINS = select_strongest_bins([parsed_missions[m] for m in bin_source_missions], N_AUDIO_BINS)
assert len(SELECTED_BINS) == N_AUDIO_BINS
print(f"[LEAKAGE CHECK] strongest-{N_AUDIO_BINS} bins computed from {len(bin_source_missions)} TRAINING missions only "
      f"(0 of {len(split_dirs['val']) + len(split_dirs['test'])} val/test missions) -> frozen for all splits: "
      f"{SELECTED_BINS.tolist()}")

# --- 3. windows (the same frozen bins for every split) ---
split_windows = {}
for split in ("train", "val", "test"):
    split_windows[split] = []
    for label in sorted(mission_splits[split]):
        for m in mission_splits[split][label]:
            if parsed_missions[m] is not None:
                split_windows[split].extend(slice_windows(parsed_missions[m], label, SELECTED_BINS))
    print(f"{split:<5}: {len(split_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
DATA_FINGERPRINT = {s: split_fingerprint(split_windows[s]) for s in split_windows}

# --- 4. datasets: sensor mean/std from the TRAINING split only, reused for val/test ---
train_ds = UAVWindowDataset(split_windows["train"])
val_ds = UAVWindowDataset(split_windows["val"], train_ds.sensor_mean, train_ds.sensor_std)
test_ds = UAVWindowDataset(split_windows["test"], train_ds.sensor_mean, train_ds.sensor_std)
_train_s = np.concatenate([w["x_s"] for w in split_windows["train"]], axis=0)
_val_s = np.concatenate([w["x_s"] for w in split_windows["val"]], axis=0)
assert val_ds.sensor_mean is train_ds.sensor_mean and test_ds.sensor_std is train_ds.sensor_std
assert np.array_equal(train_ds.sensor_mean, _train_s.mean(axis=0, keepdims=True))
print(f"[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | "
      f"train mean {np.round(train_ds.sensor_mean.ravel(), 3).tolist()} vs val's own mean "
      f"{np.round(_val_s.mean(axis=0), 3).tolist()} (not used)")
del _train_s, _val_s

# --- 5. real timestamps reach the model: tau must differ across windows ---
_tau_a = np.stack([train_ds[i]["tau_a"].numpy() for i in range(len(train_ds))])
_tau_s = np.stack([train_ds[i]["tau_s"].numpy() for i in range(len(train_ds))])
_n_unique_a, _n_unique_s = len(np.unique(_tau_a, axis=0)), len(np.unique(_tau_s, axis=0))
assert _n_unique_a > 1 and _n_unique_s > 1, "real timestamps are NOT reaching the model (tau identical for every window)"
assert not np.allclose(_tau_s, np.linspace(0.0, 1.0, TS_SENSOR)[None, :]), "tau_s equals a synthetic linspace grid"
print(f"[TIMESTAMP CHECK] {len(train_ds)} training windows: {_n_unique_a} distinct tau_a rows, {_n_unique_s} distinct "
      f"tau_s rows -> real per-window timestamps reach HTT/LACA ✅")
del _tau_a, _tau_s

Cloning https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git ...


Cloning into '/kaggle/working/tii_uav_dataset'...
Updating files: 100% (217/217), done.


Class 0 (19 flights) -> train 13, val 2, test 4
Class 1 (20 flights) -> train 14, val 3, test 3
Class 2 (20 flights) -> train 14, val 3, test 3
Class 3 (20 flights) -> train 14, val 3, test 3
Class 4 (20 flights) -> train 14, val 3, test 3


parse train:   0%|          | 0/69 [00:00<?, ?it/s]

parse val:   0%|          | 0/14 [00:00<?, ?it/s]

parse test:   0%|          | 0/16 [00:00<?, ?it/s]

[LEAKAGE CHECK] strongest-64 bins computed from 69 TRAINING missions only (0 of 30 val/test missions) -> frozen for all splits: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
train: 4934 windows | audio 64x64, sensor 128x6
val  : 954 windows | audio 64x64, sensor 128x6
test : 1130 windows | audio 64x64, sensor 128x6
[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | train mean [0.05299999937415123, -0.0430000014603138, -9.78600025177002, 0.004000000189989805, 0.003000000026077032, -0.0] vs val's own mean [0.029999999329447746, -0.061000000685453415, -9.748000144958496, 0.004999999888241291, 0.003000000026077032, -0.0] (not used)
[TIMESTAMP CHECK] 4934 training windows: 4934 distinct 

## 4. Model definition — learnable β
Encoders → HTT → bi-directional LACA → 2 Mamba blocks → attention pooling → classifier. The lag term is now **softplus(raw_β) · LN_grid(g(Δτ))**. There is one `raw_β` parameter, held by `BiModalLACAFusion`. β is computed once per forward pass and passed to both directions. The two `PairwiseLACA` modules no longer have their own β. The v18 `GridLayerNorm` and the zero-init of g are unchanged.

**Choosing raw_β's starting value.** softplus(x) = log(1 + eˣ), and its derivative is sigmoid(x). For very negative x, softplus(x) ≈ sigmoid(x) ≈ eˣ. So near zero, **dβ/draw_β ≈ β itself**: the starting gradient is about as large as the starting β.
- β(0) = 0.005 → raw_β = log(expm1(0.005)) = **−5.2958**, with **dβ/draw_β = 0.00499**. That is about 50× above the 1e-4 floor.
- A more extreme start such as raw_β = −10 gives β = 4.5e-5 and a gradient of 4.5e-5, which is **below** 1e-4.
- β = raw² started at raw = 0 has dβ/draw = 2·raw = 0, so it would never move. That form is not used.

**Why raw_β gets its own learning rate (1e-2).** AdamW takes roughly lr-sized steps whatever the gradient's size. At the shared lr of 1e-4, raw_β could move at most about 0.3 over all 20 epochs, about 3,100 steps under cosine decay. That would leave β stuck between about 0.004 and 0.007. It would be frozen by the optimizer even though its gradient is healthy. With lr = 1e-2, raw_β can move up to about 15 units:
- **up** to β ≈ 1 in about 4 epochs if the gradient keeps pointing that way, similar in pace to v18's 5-epoch warm-up but chosen by the model;
- **down** to β ≈ 1e-9 if the lag bias hurts.

Every other parameter keeps lr 1e-4. **Weight decay is 0 for raw_β.** AdamW's decay pulls raw_β toward 0, which means pulling β toward softplus(0) = 0.693, a hidden push toward a nontrivial β.

**One detail of the first training step.** At initialization g is exactly 0, so LN(g) = 0 and the loss gradient with respect to raw_β is **exactly 0 on the first batch**. After the first update g is no longer zero, and β receives gradient from then on. The first few loss gradients are printed so you can see this happen.

In [4]:
# ==============================================================================
# MODEL — encoders -> HTT (modality + continuous-time embedding of the REAL tau)
# -> bi-directional LACA (lag-aware cross-attention, lag bias softplus(raw_beta) * LN_grid(g(Δτ)):
#    ONE learnable beta shared by both directions; v18 LayerNorm + zero-init g kept; 4 heads)
# -> [H_a ; SEP ; H_s] -> 2 Mamba blocks -> attention pooling -> classifier.
# Auxiliary per-modality heads feed only the consistency loss.
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding + time embedding(tau)); modality 0 = audio, 1 = sensor."""

    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores, scaled by the shared
    learnable beta passed in by BiModalLACAFusion (v19: no per-direction beta)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n, beta):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + beta * lag_bias, dim=-1)   # beta = softplus(raw_beta), shared, 0-dim tensor
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, raw_beta_init: float = RAW_BETA_INIT):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model)
        self.laca_sa = PairwiseLACA(d_model)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))
        # v19: ONE learnable scalar for BOTH directions; beta = softplus(raw_beta) > 0 structurally. Built from a
        # constant (no RNG draws), after every RNG-drawing LACA layer.
        self.raw_beta = nn.Parameter(torch.tensor(float(raw_beta_init)))

    @property
    def beta(self) -> torch.Tensor:
        return F.softplus(self.raw_beta)

    def forward(self, Ha, Hs, tau_a, tau_s):
        beta = self.beta   # computed once, shared by both directions
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s, beta)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a, beta)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


class FallbackGatedConvBlock(nn.Module):
    """CPU-only stand-in when mamba_ssm is unavailable: depthwise causal conv + gated MLP. NOT a state-space
    model (no hidden state, no A/B/C, no discretization)."""

    def __init__(self, d_model: int, d_conv: int, expand: int):
        super().__init__()
        d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv, padding=d_conv - 1, groups=d_inner, bias=True)
        self.mixer = nn.Sequential(nn.Linear(d_inner, d_inner), nn.GELU(), nn.Linear(d_inner, d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.shape[1]
        x_branch, z_branch = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2))
        return x + self.out_proj(self.mixer(x_conv) * F.silu(z_branch))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if MAMBA_CLASS is not None:
        return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])
    return nn.ModuleList([FallbackGatedConvBlock(d_model, d_conv, expand) for _ in range(num_blocks)])


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.laca = BiModalLACAFusion(d_model)   # holds the single learnable raw_beta (no RNG draws)
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def learned_beta(self) -> float:
        """Current beta = softplus(raw_beta), as a Python float."""
        return float(self.laca.beta.detach())

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model):
    """Finalized architecture, read from the constructed modules; v19: exactly one learnable beta, shared."""
    beta_params = [n for n, p in model.named_parameters() if "beta" in n]
    assert beta_params == ["laca.raw_beta"], f"expected exactly one beta parameter, found {beta_params}"
    raw = model.laca.raw_beta
    assert raw.numel() == 1 and raw.dim() == 0 and raw.requires_grad, "raw_beta must be one learnable scalar"
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert not hasattr(laca, "beta"), f"{name} has its own beta — must use the shared one"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
        assert laca.lag_bias_norm.eps == LAG_BIAS_LN_EPS
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


_probe = AeroSyncMamba()
print(f"Model: {sum(p.numel() for p in _probe.parameters()):,} params (v18 + 1 for raw_beta) | fusion blocks: "
      f"{type(_probe.mamba_blocks[0]).__name__} x {len(_probe.mamba_blocks)} ({verify_architecture(_probe):,} params) | "
      f"LACA {LACA_NUM_HEADS} heads, g(Δτ) {LAG_BIAS_PARAMS} params, ONE shared learnable beta | backbone: {FUSION_BACKBONE}")

# --- beta initialization: value, derivative, and why this starting point ---
_raw = _probe.laca.raw_beta
(_dbeta,) = torch.autograd.grad(F.softplus(_raw), _raw)
_beta0 = float(F.softplus(_raw).detach())
print(f"[BETA INIT] raw_beta = log(expm1({BETA_INIT})) = {float(_raw.detach()):.4f} -> beta(0) = softplus(raw_beta) = "
      f"{_beta0:.6f} | d(beta)/d(raw_beta) = sigmoid(raw_beta) = {float(_dbeta):.6f} (autograd; analytic "
      f"{1 / (1 + math.exp(-RAW_BETA_INIT)):.6f})")
assert float(_dbeta) > MIN_INIT_GRAD, f"d(beta)/d(raw) {float(_dbeta):.2e} is below {MIN_INIT_GRAD}"
print(f"[BETA INIT] reasoning: for x << 0, softplus(x) ~ sigmoid(x) ~ e^x, so d(beta)/d(raw) ~ beta itself. beta(0) = "
      f"{BETA_INIT} gives a derivative ~{float(_dbeta):.4f}, {float(_dbeta) / MIN_INIT_GRAD:.0f}x the {MIN_INIT_GRAD} floor. "
      f"For comparison raw = -10 -> beta {math.log1p(math.exp(-10)):.1e}, derivative {1 / (1 + math.exp(10)):.1e} (below the "
      f"floor); beta = raw^2 at raw = 0 has derivative 2*raw = 0 (never moves) — neither is used.")
print(f"[BETA INIT] optimizer: raw_beta lr {BETA_LR} (others {LEARNING_RATE}) because AdamW steps are ~lr-sized "
      f"whatever the gradient's size: at {LEARNING_RATE}, raw_beta could move <~0.3 in 20 epochs (beta stuck at "
      f"~0.004-0.007); at {BETA_LR} it can reach beta ~1 in ~4 epochs or fall to ~1e-9. Weight decay on raw_beta = "
      f"{BETA_WEIGHT_DECAY} (decay would pull beta toward softplus(0) = 0.693).")
del _probe, _raw, _dbeta


# --- LayerNorm: unchanged from v18 (functional check + real tensor shapes) ---
_gen = torch.Generator().manual_seed(0)   # private generator: the global RNG is not touched
_x = torch.randn(3, TA_AUDIO, TS_SENSOR, generator=_gen)
_ln = GridLayerNorm()
assert _ln.eps == LAG_BIAS_LN_EPS == 1e-5 and not list(_ln.parameters())
assert torch.equal(_ln(_x), F.layer_norm(_x, (TA_AUDIO, TS_SENSOR), eps=1e-5)), "GridLayerNorm differs from v18"
print(f"[LAYERNORM] ✅ unchanged from v18: GridLayerNorm == F.layer_norm over each window's (T_m, T_n) grid, eps 1e-5, "
      f"no affine / 0 params (checked on a {tuple(_x.shape)} tensor)")
del _gen, _x, _ln
_ta = torch.stack([train_ds[i]["tau_a"] for i in range(2)])
_ts = torch.stack([train_ds[i]["tau_s"] for i in range(2)])
_probe = AeroSyncMamba()
with torch.no_grad():
    for _name, _laca, _tm, _tn in (("laca_as", _probe.laca.laca_as, _ta, _ts), ("laca_sa", _probe.laca.laca_sa, _ts, _ta)):
        _dt = (_tm.unsqueeze(2) - _tn.unsqueeze(1)).unsqueeze(-1)
        _g = _laca.lag_bias(_dt).squeeze(-1)
        _out = _laca.lag_bias_term(_tm, _tn)
        assert torch.equal(_out, F.layer_norm(_g, _g.shape[-2:], eps=LAG_BIAS_LN_EPS).unsqueeze(1))
        print(f"[LAYERNORM] {_name}: Δτ {tuple(_dt.shape)} -> g(Δτ) {tuple(_g.shape)} = (B, T_m, T_n) -> "
              f"LN over dims {tuple(_g.shape[-2:])} (per window) -> {tuple(_out.shape)} -> x softplus(raw_beta) -> "
              f"added to scores (B, {LACA_NUM_HEADS} heads, T_m, T_n)")
        assert float(_g.abs().max()) == 0.0 and float(_out.abs().max()) == 0.0, "zero-init g should give an all-zero term"
del _probe, _ta, _ts
print("[LAYERNORM] placement unchanged: PairwiseLACA.forward -> scores + beta * LN_grid(g(Δτ)), LayerNorm on g's scalar "
      "output BEFORE beta; g's final layer zero-initialized (kept from v18); beta warm-up removed.")

Model: 1,688,789 params (v18 + 1 for raw_beta) | fusion blocks: MambaBlock x 2 (874,496 params) | LACA 4 heads, g(Δτ) 97 params, ONE shared learnable beta | backbone: mamba_ssm.Mamba (real selective-scan SSM, CUDA)
[BETA INIT] raw_beta = log(expm1(0.005)) = -5.2958 -> beta(0) = softplus(raw_beta) = 0.005000 | d(beta)/d(raw_beta) = sigmoid(raw_beta) = 0.004988 (autograd; analytic 0.004988)
[BETA INIT] reasoning: for x << 0, softplus(x) ~ sigmoid(x) ~ e^x, so d(beta)/d(raw) ~ beta itself. beta(0) = 0.005 gives a derivative ~0.0050, 50x the 0.0001 floor. For comparison raw = -10 -> beta 4.5e-05, derivative 4.5e-05 (below the floor); beta = raw^2 at raw = 0 has derivative 2*raw = 0 (never moves) — neither is used.
[BETA INIT] optimizer: raw_beta lr 0.01 (others 0.0001) because AdamW steps are ~lr-sized whatever the gradient's size: at 0.0001, raw_beta could move <~0.3 in 20 epochs (beta stuck at ~0.004-0.007); at 0.01 it can reach beta ~1 in ~4 epochs or fall to ~1e-9. Weight decay on raw_

## 5. Training function
`train_one_run(seed)` works as follows.

**Before any training step:**
- resets every RNG and builds a fresh model;
- checks that the initial weights (ignoring raw_β) and the RNG state match the reused v18 β = 0 control run for that seed;
- checks that g's final layer is exactly 0;
- prints β(0) and dβ/draw_β and checks the latter is above 1e-4.

**During training:**
- raw_β has its own optimizer group (lr 1e-2, no weight decay);
- the loss gradient on raw_β is printed for the first 3 steps;
- the lowest and highest β across **every step** are tracked, and β is logged every epoch;
- epochs 1, 5, 10 and 20 are marked.

The best checkpoint includes raw_β, so the test set is evaluated with the β learned up to that checkpoint.

In [5]:
# ==============================================================================
# TRAINING — loss, metrics, learnable-beta checks (init value / derivative,
# per-step min/max, per-epoch trajectory), zero-init and pairing checks, and
# train_one_run(seed): one full model end-to-end (fresh model, AdamW with a
# separate raw_beta group, cosine schedule, 20 epochs, best-validation
# checkpoint), evaluated on the test set.
# ==============================================================================
class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}


def compute_metrics(y_true, y_prob):
    """accuracy, macro-F1, balanced accuracy, macro one-vs-rest AUROC from class probabilities (prediction = argmax)."""
    y_pred = y_prob.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_auroc": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
    }


def train_class_weights(windows):
    """Inverse class frequency over the TRAINING windows."""
    counts = np.bincount([w["label"] for w in windows], minlength=N_CLASSES)
    return torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)


def _batch_to_device(batch, non_blocking=False):
    return tuple(batch[k].to(device, non_blocking=non_blocking) for k in ("x_a", "x_s", "tau_a", "tau_s", "label"))


def predict_probs(model, loader):
    """Softmax probabilities and true labels over a loader (eval mode, no grad), in loader order."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits, _, _, _ = model(xa, xs, ta, ts)
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
CLASS_WEIGHTS = train_class_weights(split_windows["train"]).to(device)
TEST_LABELS = np.array([w["label"] for w in split_windows["test"]], dtype=np.int64)   # test-window order, fixed


# Fixed probe batch for the lag-contribution check: timestamps of the first BATCH_SIZE training windows, read by
# index (no DataLoader, no RNG). The lag-bias term depends only on the timestamps.
LAG_PROBE_TAUS = tuple(torch.stack([train_ds[i][k] for i in range(min(BATCH_SIZE, len(train_ds)))])
                       for k in ("tau_a", "tau_s"))


def lag_bias_contribution(model):
    """Per LACA direction: max|beta * LN_grid(g(Δτ))|, the exact term PairwiseLACA adds to the attention scores
    (same lag_bias_term method, current learned beta), plus the raw std of g before the LayerNorm."""
    ta, ts = (t.to(device) for t in LAG_PROBE_TAUS)
    out = {}
    with torch.no_grad():
        beta = model.laca.beta
        for name, laca, tau_m, tau_n in (("laca_as", model.laca.laca_as, ta, ts), ("laca_sa", model.laca.laca_sa, ts, ta)):
            contrib = beta * laca.lag_bias_term(tau_m, tau_n)
            g_raw = laca.lag_bias((tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)).squeeze(-1)
            out[name] = {"contrib_absmax": float(contrib.abs().max()), "g_raw_std": float(g_raw.std())}
    return out


def g_final_layer_absmax(model):
    """max|weight| and max|bias| of g's final Linear(32, 1), per LACA direction."""
    return {name: (float(getattr(model.laca, name).lag_bias[-1].weight.detach().abs().max()),
                   float(getattr(model.laca, name).lag_bias[-1].bias.detach().abs().max()))
            for name in ("laca_as", "laca_sa")}


def weights_sha1(model, exclude=()):
    return hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                 for k, v in model.state_dict().items() if k not in exclude)).hexdigest()


def build_optimizer(model):
    """AdamW: every parameter at LEARNING_RATE / WEIGHT_DECAY as before, except raw_beta (BETA_LR, no decay)."""
    raw = model.laca.raw_beta
    others = [p for p in model.parameters() if p is not raw]
    assert len(others) + 1 == len(list(model.parameters()))
    return torch.optim.AdamW([{"params": others, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
                              {"params": [raw], "lr": BETA_LR, "weight_decay": BETA_WEIGHT_DECAY}])


def train_one_run(seed: int):
    """Train one learnable-beta model with `seed` applied to every RNG, evaluate it on the test set.
    Returns (trained model on CPU, result dict incl. test probabilities)."""
    t0 = time.time()
    tag = f"[seed={seed} learnable-beta]"
    set_all_seeds(seed)
    model = AeroSyncMamba().to(device)
    init_sha1 = weights_sha1(model)
    init_sha1_excl = weights_sha1(model, exclude=("laca.raw_beta",))
    rng_sha1 = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()
    mamba_params = verify_architecture(model)
    print(f"{tag} one shared learnable beta: {[n for n, _ in model.named_parameters() if 'beta' in n]} | "
          f"fusion {type(model.mamba_blocks[0]).__name__} ({mamba_params:,} params)")

    # --- pairing with the REUSED v18 beta=0 control: same initial weights (raw_beta excluded) + same RNG stream ---
    ctrl = V18_BETA0_CONTROL[seed]
    paired_ok = init_sha1_excl[:12] == ctrl["init_sha1"] and rng_sha1[:12] == ctrl["rng_sha1"]
    print(f"{tag} [PAIRING] initial weights excl. raw_beta {init_sha1_excl[:12]} vs control {ctrl['init_sha1']} | torch "
          f"RNG after construction {rng_sha1[:12]} vs control {ctrl['rng_sha1']} -> "
          + ("✅ same starting point and RNG stream as the reused beta=0 run" if paired_ok else
             "🚨 MISMATCH — this run does not start where its reused control started; the pairing is broken"))

    # --- ZERO-INIT (kept from v18): before any training step ---
    zi = g_final_layer_absmax(model)
    assert all(w == 0.0 and b == 0.0 for w, b in zi.values()), f"g final layer is not zero-initialized: {zi}"
    print(f"{tag} [ZERO-INIT] ✅ g final layer max|W| / max|b| = "
          + " | ".join(f"{n}: {w!r} / {b!r}" for n, (w, b) in zi.items()) + " -> g(Δτ) = 0 for every Δτ")

    # --- beta(0) and d(beta)/d(raw_beta) at initialization ---
    raw = model.laca.raw_beta
    (dbeta_draw,) = torch.autograd.grad(F.softplus(raw), raw)
    beta0, dbeta_draw, raw0 = model.learned_beta(), float(dbeta_draw), float(raw.detach())
    assert dbeta_draw > MIN_INIT_GRAD, f"d(beta)/d(raw_beta) = {dbeta_draw:.2e} <= {MIN_INIT_GRAD}: beta effectively frozen"
    print(f"{tag} [BETA INIT] raw_beta {raw0:.4f} | beta(0) = {beta0:.6f} | d(beta)/d(raw_beta) = {dbeta_draw:.6f} "
          f"(> {MIN_INIT_GRAD} ✅ — {dbeta_draw / MIN_INIT_GRAD:.0f}x the floor)")

    criterion = AeroSyncLoss(CLASS_WEIGHTS)
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

    beta_history, lag_history, grad_history = {}, {}, {}
    beta_min, beta_max, first_grads = beta0, beta0, []
    best_val_f1, best_state, best_epoch, best_beta = 0.0, None, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = defaultdict(float)
        grad_abs_sum = 0.0
        for batch in tqdm(train_loader, desc=f"seed {seed} learnable-beta epoch {epoch:02d}/{EPOCHS}", leave=False):
            xa, xs, ta, ts, y = _batch_to_device(batch, non_blocking=True)
            optimizer.zero_grad()
            logits, pooled, aux_logits, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux_logits)
            loss.backward()
            g_raw = float(raw.grad) if raw.grad is not None else 0.0   # dL/d(raw_beta), before clipping
            grad_abs_sum += abs(g_raw)
            if len(first_grads) < 3:
                first_grads.append(g_raw)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            b_now = model.learned_beta()
            beta_min, beta_max = min(beta_min, b_now), max(beta_max, b_now)
            running["loss"] += loss.item()
            for k, v in parts.items():
                running[k] += v
        scheduler.step()
        if epoch == 1:
            print(f"{tag} [BETA GRAD] dL/d(raw_beta) on the first 3 steps: " + ", ".join(f"{g:+.3e}" for g in first_grads)
                  + " (step 1 is exactly 0 by construction: zero-init g -> LN(g) = 0; nonzero once g has moved)")

        beta_history[epoch] = model.learned_beta()
        grad_history[epoch] = grad_abs_sum / len(train_loader)
        lag = lag_bias_contribution(model)
        lag_history[epoch] = max(r["contrib_absmax"] for r in lag.values())

        val_probs, val_labels = predict_probs(model, val_loader)
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch, best_beta = val_f1, copy.deepcopy(model.state_dict()), epoch, beta_history[epoch]
        n = len(train_loader)
        print(f"{tag} epoch {epoch:02d}/{EPOCHS} | beta {beta_history[epoch]:.5f} | mean |dL/draw_beta| "
              f"{grad_history[epoch]:.2e} | max|beta*LN(g)| {lag_history[epoch]:.4f} | train loss {running['loss'] / n:.4f} "
              f"(cls {running['loss_cls'] / n:.4f}, align {running['loss_align'] / n:.4f}, cons {running['loss_cons'] / n:.4f})"
              f" | val macro-F1 {val_f1:.4f}{'  * best' if improved else ''}")
        if epoch in BETA_CHECKPOINT_EPOCHS:
            print(f"{tag} [BETA] epoch {epoch}: beta = {beta_history[epoch]:.5f} (raw_beta {float(raw.detach()):+.4f})")

    assert beta_min >= 0.0, f"beta went negative ({beta_min}) — impossible under softplus; something is wrong"
    print(f"{tag} [BETA TRAJECTORY] beta(0) {beta0:.5f} -> "
          + " -> ".join(f"ep{e} {beta_history[e]:.5f}" for e in BETA_CHECKPOINT_EPOCHS)
          + f" | min over all {EPOCHS * len(train_loader)} steps {beta_min:.3e} (>= 0 ✅), max {beta_max:.5f}")
    beta_final, raw_final, lag_final = beta_history[EPOCHS], float(raw.detach()), lag_history[EPOCHS]

    model.load_state_dict(best_state)   # includes raw_beta -> the checkpoint's own learned beta
    assert abs(model.learned_beta() - best_beta) < 1e-7
    lag_best = max(r["contrib_absmax"] for r in lag_bias_contribution(model).values())
    test_probs, test_labels = predict_probs(model, test_loader)
    metrics = compute_metrics(test_labels, test_probs)
    run_name = f"learnable_beta_seed{seed}"
    torch.save(best_state, CHECKPOINT_DIR / f"{run_name}_best.pt")
    np.save(PROBS_DIR / f"{run_name}_test_probs.npy", test_probs)
    np.save(PROBS_DIR / f"{run_name}_test_labels.npy", test_labels)

    result = {
        "seed": seed,
        **metrics,
        "best_val_macro_f1": round(best_val_f1, 6),
        "best_epoch": best_epoch,
        "n_test_windows": len(test_labels),
        "init_sha1": init_sha1[:12],
        "init_sha1_excl_raw_beta": init_sha1_excl[:12],
        "rng_sha1": rng_sha1[:12],
        "paired_with_control": paired_ok,
        "raw_beta_init": raw0,
        "beta_init": beta0,
        "dbeta_draw_init": dbeta_draw,
        **{f"beta_epoch{e}": beta_history[e] for e in BETA_CHECKPOINT_EPOCHS},
        "beta_final": beta_final,
        "beta_at_best": best_beta,
        "raw_beta_final": raw_final,
        "beta_min_all_steps": beta_min,
        "beta_max_all_steps": beta_max,
        "lag_contrib_final_epoch": lag_final,
        "lag_contrib_best": lag_best,
        "beta_history": beta_history,
        "train_seconds": round(time.time() - t0, 1),
        "test_probs": test_probs,
        "test_labels": test_labels,
    }
    return model.cpu(), result

## 6. Training driver: 5 learnable-β runs
One run per seed: 42, 123, 2024, 7, 99. The β = 0.0 control is **reused from v18, not retrained**. After every run the driver:
- appends a row to `/kaggle/working/beta_learnable_paired_results.csv`;
- prints the paired difference against the reused control for that seed;
- frees the GPU and prints a progress line.

In [6]:
# ==============================================================================
# DRIVER — one learnable-beta run per seed. The beta=0.0 control is REUSED from
# v18 (V18_BETA0_CONTROL, Section 2) — NOT retrained. After every run: metrics,
# paired diff vs the reused control, CSV row, GPU freed, progress.
# ==============================================================================
import pandas as pd

RESULTS_CSV_FIELDS = (["run", "seed", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc", "control_macro_f1",
                       "paired_diff_macro_f1", "best_val_macro_f1", "best_epoch", "n_test_windows", "init_sha1",
                       "init_sha1_excl_raw_beta", "rng_sha1", "paired_with_control", "raw_beta_init", "beta_init",
                       "dbeta_draw_init"] + [f"beta_epoch{e}" for e in BETA_CHECKPOINT_EPOCHS]
                      + ["beta_final", "beta_at_best", "raw_beta_final", "beta_min_all_steps", "beta_max_all_steps",
                         "lag_contrib_final_epoch", "lag_contrib_best", "train_seconds", "finished_at"])
METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]

print("CONTROL GROUP: REUSING v18's 5 beta=0.0 runs as-is — NOT retraining them. That condition is architecturally")
print("this experiment's baseline (v18 LayerNorm + zero-init g, beta hard-fixed at exactly 0 at every epoch, so the")
print("warm-up was a no-op; lag term asserted exactly 0 each epoch) and it reproduced v17's beta=0.0 runs exactly.")
print("  control macro-F1: " + ", ".join(f"seed {s}: {V18_BETA0_CONTROL[s]['macro_f1']:.4f}" for s in SEEDS))

if RESULTS_CSV.exists():
    RESULTS_CSV.rename(RESULTS_CSV.with_suffix(f".{datetime.now():%Y%m%d_%H%M%S}.bak.csv"))

run_results = {}
t_driver = time.time()
for i, seed in enumerate(SEEDS, start=1):
    print("\n" + "=" * 100 + f"\nRUN {i}/{len(SEEDS)} — seed={seed}, LEARNABLE beta (init {BETA_INIT})\n" + "=" * 100)
    model, result = train_one_run(seed)
    ctrl_f1 = V18_BETA0_CONTROL[seed]["macro_f1"]
    result["control_macro_f1"] = ctrl_f1
    result["paired_diff_macro_f1"] = result["macro_f1"] - ctrl_f1
    run_results[seed] = result
    print(f"[seed={seed} learnable-beta] TEST: accuracy {result['accuracy']:.4f} | macro-F1 {result['macro_f1']:.4f} | "
          f"balanced accuracy {result['balanced_accuracy']:.4f} | macro-AUROC {result['macro_auroc']:.4f} | best epoch "
          f"{result['best_epoch']} | {result['train_seconds']:.0f}s")
    print(f"[PAIR seed={seed}] learnable beta {result['macro_f1']:.4f} vs reused beta=0 control {ctrl_f1:.4f} -> paired diff "
          f"{result['paired_diff_macro_f1']:+.4f} | learned beta final {result['beta_final']:.5f} (at best ckpt "
          f"{result['beta_at_best']:.5f})")

    write_header = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_CSV_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow({"run": i, **{k: result[k] for k in RESULTS_CSV_FIELDS if k in result},
                         "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    elapsed = time.time() - t_driver
    print(f"\n>>> {i}/{len(SEEDS)} complete: seed={seed}, learnable beta finished | elapsed {elapsed / 60:.1f} min | "
          f"est. remaining {elapsed / i * (len(SEEDS) - i) / 60:.1f} min")

print(f"\nAll {len(SEEDS)} learnable-beta runs finished in {(time.time() - t_driver) / 60:.1f} min. Results: {RESULTS_CSV}")

CONTROL GROUP: REUSING v18's 5 beta=0.0 runs as-is — NOT retraining them. That condition is architecturally
this experiment's baseline (v18 LayerNorm + zero-init g, beta hard-fixed at exactly 0 at every epoch, so the
warm-up was a no-op; lag term asserted exactly 0 each epoch) and it reproduced v17's beta=0.0 runs exactly.
  control macro-F1: seed 42: 0.9057, seed 123: 0.8827, seed 2024: 0.8764, seed 7: 0.9019, seed 99: 0.8638

RUN 1/5 — seed=42, LEARNABLE beta (init 0.005)
[seed=42 learnable-beta] one shared learnable beta: ['laca.raw_beta'] | fusion MambaBlock (874,496 params)
[seed=42 learnable-beta] [PAIRING] initial weights excl. raw_beta 5850841e699f vs control 5850841e699f | torch RNG after construction 5641b401720d vs control 5641b401720d -> ✅ same starting point and RNG stream as the reused beta=0 run
[seed=42 learnable-beta] [ZERO-INIT] ✅ g final layer max|W| / max|b| = laca_as: 0.0 / 0.0 | laca_sa: 0.0 / 0.0 -> g(Δτ) = 0 for every Δτ
[seed=42 learnable-beta] [BETA INIT] raw_

seed 42 learnable-beta epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

[seed=42 learnable-beta] [BETA GRAD] dL/d(raw_beta) on the first 3 steps: +0.000e+00, -3.992e-08, -1.393e-08 (step 1 is exactly 0 by construction: zero-init g -> LN(g) = 0; nonzero once g has moved)
[seed=42 learnable-beta] epoch 01/20 | beta 0.00783 | mean |dL/draw_beta| 2.65e-05 | max|beta*LN(g)| 0.0182 | train loss 1.4174 (cls 1.0707, align 3.1025, cons 0.3650) | val macro-F1 0.8272  * best
[seed=42 learnable-beta] [BETA] epoch 1: beta = 0.00783 (raw_beta -4.8462)


seed 42 learnable-beta epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

[seed=42 learnable-beta] epoch 02/20 | beta 0.00994 | mean |dL/draw_beta| 1.25e-04 | max|beta*LN(g)| 0.0244 | train loss 0.9259 (cls 0.5977, align 2.6650, cons 0.6171) | val macro-F1 0.6814


...

[seed=99 learnable-beta] epoch 18/20 | beta 0.00477 | mean |dL/draw_beta| 1.40e-04 | max|beta*LN(g)| 0.0123 | train loss 0.3208 (cls 0.1494, align 1.2800, cons 0.4338) | val macro-F1 0.8413


seed 99 learnable-beta epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

[seed=99 learnable-beta] epoch 19/20 | beta 0.00476 | mean |dL/draw_beta| 1.42e-04 | max|beta*LN(g)| 0.0123 | train loss 0.2899 (cls 0.1183, align 1.2787, cons 0.4380) | val macro-F1 0.8781


seed 99 learnable-beta epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

[seed=99 learnable-beta] epoch 20/20 | beta 0.00477 | mean |dL/draw_beta| 1.33e-04 | max|beta*LN(g)| 0.0123 | train loss 0.2904 (cls 0.1198, align 1.2685, cons 0.4370) | val macro-F1 0.8863
[seed=99 learnable-beta] [BETA] epoch 20: beta = 0.00477 (raw_beta -5.3436)
[seed=99 learnable-beta] [BETA TRAJECTORY] beta(0) 0.00500 -> ep1 0.00391 -> ep5 0.00309 -> ep10 0.00334 -> ep20 0.00477 | min over all 3100 steps 2.561e-03 (>= 0 ✅), max 0.00501
[seed=99 learnable-beta] TEST: accuracy 0.8690 | macro-F1 0.8638 | balanced accuracy 0.8598 | macro-AUROC 0.9830 | best epoch 16 | 213s
[PAIR seed=99] learnable beta 0.8638 vs reused beta=0 control 0.8638 -> paired diff +0.0000 | learned beta final 0.00477 (at best ckpt 0.00477)

>>> 5/5 complete: seed=99, learnable beta finished | elapsed 28.6 min | est. remaining 0.0 min

All 5 learnable-beta runs finished in 28.6 min. Results: /kaggle/working/beta_learnable_paired_results.csv


## 7. Paired statistics, all β approaches side by side, and where β converged
This section reads `beta_learnable_paired_results.csv` and pairs each seed with its **reused** v18 β = 0.0 control.

**Accuracy statistics** (same as v17/v18): a per-seed paired table for all four metrics, then for macro-F1 the mean paired difference, the std of the paired differences, the win count, a **paired t-test (primary)** and a Wilcoxon signed-rank test (secondary).

**Why the t-test is primary.** With 5 pairs, the exact two-sided Wilcoxon p-value can never go below 2/2⁵ = 0.0625, so it can never reach 0.05. The t-test uses the size of each difference. It assumes the differences are roughly normal, which can't be checked with 5 pairs, so the Wilcoxon test is reported as a robustness check.

**All β approaches.** One row per approach, on the same seeds: mean macro-F1, and for each β approach its mean paired difference, std, win count and p-value against β = 0.0.

**Where β converged.** Each seed's β path: β(0), epochs 1, 5, 10 and 20, the best checkpoint, and the lowest and highest values reached. Classification:
- **NEAR-ZERO**: every seed's final β ≤ 0.05. The lag term then adds at most about 0.1 to any attention score, which is negligible.
- **CONSISTENTLY NONTRIVIAL**: every seed's final β ≥ 0.2, and the largest is at most 3× the smallest.
- **INCONSISTENT / SEED-DEPENDENT**: anything else.
- **β DID NOT MOVE**: no seed's β changed by more than 1.5× from its start. The test would then be inconclusive, and the notebook says so.

In [7]:
# ==============================================================================
# ANALYSIS — paired stats vs the reused control, all approaches side by side,
# and where beta converged
# ==============================================================================
from scipy import stats

df = pd.read_csv(RESULTS_CSV)
done = [s for s in SEEDS if s in set(df["seed"])]
n = len(done)
print(f"Learnable-beta runs finished: {n}/{len(SEEDS)} -> seeds {done} (each paired with its REUSED v18 beta=0.0 control)")
if n < len(SEEDS):
    print(f"⚠️  PARTIAL RESULT: only {n} of {len(SEEDS)} seeds finished — everything below uses those seeds only.")
assert n >= 2, "need at least 2 finished seeds for any paired statistic"
row = {s: df[df["seed"] == s].iloc[-1] for s in done}
ctrl = {m: np.array([V18_BETA0_CONTROL[s][m] for s in done]) for m in METRICS}
learn = {m: np.array([float(row[s][m]) for s in done]) for m in METRICS}


def paired_stats(b1, b0):
    d = np.asarray(b1, dtype=float) - np.asarray(b0, dtype=float)
    k = len(d)
    out = {"d": d, "n": k, "mean": float(d.mean()), "sd": float(d.std(ddof=1)), "wins": int((d > 0).sum()),
           "losses": int((d < 0).sum()), "ties": int((d == 0).sum())}
    out["se"] = out["sd"] / math.sqrt(k)
    tr = stats.ttest_rel(b1, b0)
    out["t"], out["p_t"] = float(tr.statistic), float(tr.pvalue)
    tc = float(stats.t.ppf(0.975, df=k - 1))
    out["ci"] = (out["mean"] - tc * out["se"], out["mean"] + tc * out["se"])
    try:
        out["p_w"] = float(stats.wilcoxon(d, alternative="two-sided", method="exact").pvalue)
    except Exception:
        out["p_w"] = float("nan")
    return out


# --- 1. per-seed paired tables, all 4 metrics ---
for m in METRICS:
    d = learn[m] - ctrl[m]
    print(f"\n{'PRIMARY — ' if m == 'macro_f1' else ''}{m}")
    print(f"  {'seed':>6} {'beta=0 (v18)':>13} {'learnable':>10} {'diff':>9}  favors")
    for s, c, l, x in zip(done, ctrl[m], learn[m], d):
        print(f"  {s:>6} {c:>13.4f} {l:>10.4f} {x:>+9.4f}  {'learnable' if x > 0 else ('beta=0' if x < 0 else 'tie')}")
    print(f"  {'mean':>6} {ctrl[m].mean():>13.4f} {learn[m].mean():>10.4f} {d.mean():>+9.4f}  "
          f"(std of paired diffs {d.std(ddof=1):.4f}; learnable wins {int((d > 0).sum())}/{n})")

# --- 2. paired statistics on macro-F1 ---
P = paired_stats(learn["macro_f1"], ctrl["macro_f1"])
mean_d, sd_d, wins, p_t = P["mean"], P["sd"], P["wins"], P["p_t"]
print("\n" + "=" * 100)
print(f"PAIRED STATISTICS — test macro-F1, learnable beta minus REUSED beta=0.0 control, {n} matched seeds")
print("=" * 100)
print(f"  paired differences          : " + ", ".join(f"{s}: {x:+.4f}" for s, x in zip(done, P["d"])))
print(f"  mean paired difference      : {mean_d:+.4f}")
print(f"  std of paired differences   : {sd_d:.4f}   (sample std, ddof=1)")
print(f"  95% CI of mean difference   : [{P['ci'][0]:+.4f}, {P['ci'][1]:+.4f}]")
print(f"  win count                   : learnable better on {wins}/{n} seeds (beta=0 better on {P['losses']}, ties {P['ties']})")
print(f"  paired t-test (PRIMARY)     : t = {P['t']:+.3f}, df = {n - 1}, two-sided p = {p_t:.4f}")
print(f"  Wilcoxon signed-rank (exact): two-sided p = {P['p_w']:.4f}   (smallest achievable with n={n}: {2 / 2 ** n:.4f})")

# --- 3. all beta approaches, one row each (same seeds) ---
APPROACHES = [
    ("beta=0.0 (fixed, no lag bias)", {s: V18_BETA0_CONTROL[s]["macro_f1"] for s in done}),
    ("beta=1.0 unstabilized (v17)", {s: V17_BETA1_MACRO_F1[s] for s in done}),
    ("beta=1.0 stabilized+warmup (v18)", {s: V18_BETA1_MACRO_F1[s] for s in done}),
    ("beta learnable, non-negative, init~0 (v19)", {s: float(row[s]["macro_f1"]) for s in done}),
]
print("\n" + "=" * 118)
print(f"ALL BETA APPROACHES — test macro-F1 on seeds {done}; paired statistics vs beta=0.0 (same seeds, same control)")
print("=" * 118)
print(f"  {'approach':<44} {'mean F1':>8} {'mean diff':>10} {'std diff':>9} {'wins vs b=0':>12} {'t-test p':>9}")
summary_rows = {}
for name, vals in APPROACHES:
    v = np.array([vals[s] for s in done])
    if name.startswith("beta=0.0"):
        print(f"  {name:<44} {v.mean():>8.4f} {'—':>10} {'—':>9} {'—':>12} {'—':>9}")
        continue
    ps = paired_stats(v, ctrl["macro_f1"])
    summary_rows[name] = ps
    print(f"  {name:<44} {v.mean():>8.4f} {ps['mean']:>+10.4f} {ps['sd']:>9.4f} {str(ps['wins']) + '/' + str(n):>12} "
          f"{ps['p_t']:>9.4f}")

# --- 4. where did beta converge? ---
B = {s: {"init": float(row[s]["beta_init"]), **{e: float(row[s][f"beta_epoch{e}"]) for e in BETA_CHECKPOINT_EPOCHS},
         "best": float(row[s]["beta_at_best"]), "min": float(row[s]["beta_min_all_steps"]),
         "max": float(row[s]["beta_max_all_steps"]), "dinit": float(row[s]["dbeta_draw_init"]),
         "lag": float(row[s]["lag_contrib_final_epoch"])} for s in done}
print("\n" + "=" * 118)
print("WHERE DID BETA CONVERGE? — learned beta = softplus(raw_beta) per seed")
print("=" * 118)
print(f"  {'seed':>6} {'beta(0)':>8} " + " ".join(f"{'ep' + str(e):>8}" for e in BETA_CHECKPOINT_EPOCHS)
      + f" {'best ckpt':>9} {'min':>9} {'max':>8} {'x init':>8} {'lag term':>8} {'F1 diff':>8}")
for s, x in zip(done, P["d"]):
    b = B[s]
    print(f"  {s:>6} {b['init']:>8.5f} " + " ".join(f"{b[e]:>8.5f}" for e in BETA_CHECKPOINT_EPOCHS)
          + f" {b['best']:>9.5f} {b['min']:>9.2e} {b['max']:>8.4f} {b[EPOCHS] / b['init']:>8.2f} {b['lag']:>8.4f} {x:>+8.4f}")
finals = np.array([B[s][EPOCHS] for s in done])
all_min = min(B[s]["min"] for s in done)
print(f"\n  beta never negative: minimum over every training step of every run = {all_min:.3e} -> "
      f"{'✅ >= 0 (softplus guarantee holds)' if all_min >= 0 else '🚨 NEGATIVE'}")
print(f"  d(beta)/d(raw_beta) at init: " + ", ".join(f"{s}: {B[s]['dinit']:.5f}" for s in done)
      + f" -> all > {MIN_INIT_GRAD}: {'✅' if all(B[s]['dinit'] > MIN_INIT_GRAD for s in done) else '❌'}")
print(f"  final beta across seeds: min {finals.min():.5f} | max {finals.max():.5f} | mean {finals.mean():.5f} | "
      f"max/min {finals.max() / finals.min():.2f} | CV {finals.std(ddof=1) / finals.mean():.2f}")
print(f"  lag-term size max|beta*LN(g)| at the final epoch: {min(B[s]['lag'] for s in done):.4f}-"
      f"{max(B[s]['lag'] for s in done):.4f} (v17 fixed beta=1: {min(V17_BETA1_LAG_CONTRIB.values()):.2f}-"
      f"{max(V17_BETA1_LAG_CONTRIB.values()):.2f}; v18 fixed beta=1: {min(V18_BETA1_LAG_CONTRIB.values()):.2f}-"
      f"{max(V18_BETA1_LAG_CONTRIB.values()):.2f})")

moved = [abs(math.log(B[s][EPOCHS] / B[s]["init"])) > math.log(1.5) for s in done]
n_near_zero = int((finals <= BETA_NEAR_ZERO).sum())
n_nontrivial = int((finals >= BETA_NONTRIVIAL).sum())
n_grew = int(sum(B[s][EPOCHS] > B[s]["init"] for s in done))
if not any(moved):
    BETA_CONVERGENCE = "DID NOT MOVE"
    beta_msg = (f"no seed's beta changed by more than 1.5x from its start ({BETA_INIT}). The learnability test is "
                f"INCONCLUSIVE: the model did not use its freedom to change beta. This does not show the lag bias is "
                f"useless; beta may simply not have been able to move.")
elif n_near_zero == n:
    BETA_CONVERGENCE = "CONSISTENTLY NEAR-ZERO"
    beta_msg = (f"all {n} seeds ended with beta <= {BETA_NEAR_ZERO} (max {finals.max():.5f}); {n - n_grew} of {n} "
                f"ended below their starting value. Every seed chose to leave the lag bias (nearly) off, which "
                f"matches the beta=0.0 results from v17/v18: the model does not find the lag bias useful.")
elif n_nontrivial == n and finals.max() / finals.min() <= BETA_CONSISTENT_MAXMIN:
    BETA_CONVERGENCE = "CONSISTENTLY NONTRIVIAL"
    beta_msg = (f"all {n} seeds grew beta to >= {BETA_NONTRIVIAL} ({finals.min():.3f}-{finals.max():.3f}, max/min "
                f"{finals.max() / finals.min():.2f} <= {BETA_CONSISTENT_MAXMIN}). Every seed chose to use the lag bias, "
                f"at a similar strength: the model selects a real, stable use for the mechanism.")
else:
    BETA_CONVERGENCE = "INCONSISTENT / SEED-DEPENDENT"
    beta_msg = (f"final beta ranges {finals.min():.5f}-{finals.max():.5f} (max/min {finals.max() / finals.min():.1f}): "
                f"{n_near_zero} seed(s) near zero (<= {BETA_NEAR_ZERO}), {n_nontrivial} nontrivial (>= {BETA_NONTRIVIAL}), "
                f"{n - n_near_zero - n_nontrivial} in between; {n_grew}/{n} grew. Where beta lands depends on the "
                f"seed. This is the same instability seen in v17 (lag-term size 0.23-0.68 across seeds), now showing "
                f"up in the learned beta instead of in g's output size.")
rho = stats.spearmanr(finals, P["d"]).statistic if n >= 3 else float("nan")

print("\n" + "#" * 118)
print(f"#  BETA CONVERGENCE: {BETA_CONVERGENCE}")
print(f"#  {beta_msg}")
print(f"#  descriptive only (n={n}): Spearman rank correlation between final beta and the per-seed macro-F1 gain = "
      f"{rho:+.2f}")
print("#" * 118)

# --- 5. accuracy verdict (same rule as v17/v18) ---
exceeds_noise = abs(mean_d) > sd_d
significant = p_t < PAIRED_ALPHA
learn_consistent = mean_d > 0 and wins >= n - 1
ctrl_consistent = mean_d < 0 and P["losses"] >= n - 1
if learn_consistent and (exceeds_noise or significant):
    VERDICT = "WELL-SUPPORTED"
elif ctrl_consistent and (exceeds_noise or significant):
    VERDICT = "REVERSED (beta=0.0 better)"
else:
    VERDICT = "NOT WELL-SUPPORTED"
print(f"#  ACCURACY VERDICT (same rule as v17/v18): learnable beta's advantage over beta=0.0 is {VERDICT}")
print(f"#  wins {wins}/{n} (need >= {n - 1}) | |mean| {abs(mean_d):.4f} vs std {sd_d:.4f} | paired t-test p = {p_t:.4f}")
if VERDICT == "WELL-SUPPORTED":
    print(f"#  Learnable beta beats no lag bias on {wins}/{n} seeds by {mean_d:+.4f} on average, more than the paired "
          f"noise ({sd_d:.4f}).")
elif VERDICT.startswith("REVERSED"):
    print(f"#  No lag bias beats learnable beta on {P['losses']}/{n} seeds by {-mean_d:.4f} on average, more than the "
          f"paired noise ({sd_d:.4f}): even a self-selected beta hurts.")
else:
    print(f"#  Learnable beta does not reliably beat no lag bias: mean {mean_d:+.4f} vs paired std {sd_d:.4f}, wins "
          f"{wins}/{n}, p = {p_t:.4f}.")
if BETA_CONVERGENCE == "CONSISTENTLY NEAR-ZERO" and VERDICT == "NOT WELL-SUPPORTED":
    print("#  Together: the model switched the lag bias off, and its accuracy matches beta=0.0. Both results point the same way.")
if n < len(SEEDS):
    print(f"#  ⚠️  based on {n}/{len(SEEDS)} seeds only.")
print("#" * 118)

Learnable-beta runs finished: 5/5 -> seeds [42, 123, 2024, 7, 99] (each paired with its REUSED v18 beta=0.0 control)

accuracy
    seed  beta=0 (v18)  learnable      diff  favors
      42        0.9088     0.9080   -0.0009  beta=0
     123        0.8876     0.8876   +0.0000  tie
    2024        0.8805     0.8814   +0.0009  learnable
       7        0.9062     0.9062   +0.0000  learnable
      99        0.8690     0.8690   +0.0000  tie
    mean        0.8904     0.8904   +0.0000  (std of paired diffs 0.0006; learnable wins 2/5)

PRIMARY — macro_f1
    seed  beta=0 (v18)  learnable      diff  favors
      42        0.9057     0.9048   -0.0009  beta=0
     123        0.8827     0.8827   +0.0000  tie
    2024        0.8764     0.8772   +0.0009  learnable
       7        0.9019     0.9019   +0.0000  tie
      99        0.8638     0.8638   +0.0000  tie
    mean        0.8861     0.8861   -0.0000  (std of paired diffs 0.0006; learnable wins 1/5)

balanced_accuracy
    seed  beta=0 (v18)  lear

## 8. Final validation
A recap of every verification check: data and RNG diagnostics, pairing with the reused control, zero-init, β(0) and its derivative, β never negative, the LayerNorm unchanged from v18, and both conclusions.

In [8]:
# ==============================================================================
# FINAL VALIDATION — every verification hook in one place
# ==============================================================================
print("=" * 100)
print("FINAL VALIDATION")
print("=" * 100)
data_checks = {
    "strongest-64 bin indices": SELECTED_BINS.tolist() == REFERENCE_DATA["bin_indices"],
    **{f"{s} windows / sensor values / timestamps / class counts": DATA_FINGERPRINT[s] ==
       REFERENCE_DATA["sensor_fingerprint"][s] for s in ("train", "val", "test")},
    "seed-42 torch RNG after construction == v16/v18 (no extra draws)":
        run_results[42]["rng_sha1"].startswith(REFERENCE_SEED42_RNG_PREFIX) if 42 in run_results else False,
    "fusion backbone is real mamba_ssm": MAMBA_CLASS is not None,
}
for name, ok in data_checks.items():
    print(f"  {name:<64}: {'match' if ok else 'MISMATCH'}")

pair_ok = {s: r["paired_with_control"] for s, r in run_results.items()}
print("\n" + "=" * 100)
print(f"  control group                                   : v18 beta=0.0 rows REUSED, not retrained ({len(SEEDS)} seeds)")
print(f"  each run starts where its reused control started: {'✅ all seeds' if all(pair_ok.values()) else '🚨 ' + str(pair_ok)}")
print(f"  zero-init g final layer before training         : ✅ asserted for all {len(run_results)} runs")
print(f"  beta(0) / d(beta)/d(raw) at init                : " + ", ".join(
      f"{s}: {r['beta_init']:.5f} / {r['dbeta_draw_init']:.5f}" for s, r in run_results.items())
      + f"  (all > {MIN_INIT_GRAD}: {'✅' if all(r['dbeta_draw_init'] > MIN_INIT_GRAD for r in run_results.values()) else '❌'})")
print(f"  beta never negative (min over every step)       : {min(r['beta_min_all_steps'] for r in run_results.values()):.3e} "
      f"{'✅' if all(r['beta_min_all_steps'] >= 0 for r in run_results.values()) else '❌'}")
print(f"  LayerNorm on g(Δτ) unchanged from v18           : ✅ (functional check + shapes in Section 4)")
print(f"  beta at epochs {BETA_CHECKPOINT_EPOCHS}              : " + " | ".join(
      f"{s}: " + "/".join(f"{r[f'beta_epoch{e}']:.4f}" for e in BETA_CHECKPOINT_EPOCHS) for s, r in run_results.items()))
print(f"  BETA CONVERGENCE                                : {BETA_CONVERGENCE}")
print(f"  ACCURACY VERDICT                                : learnable beta vs beta=0.0 {VERDICT} — mean diff "
      f"{mean_d:+.4f}, std {sd_d:.4f}, wins {wins}/{n}, p = {p_t:.4f}")
print("=" * 100)
if not all(pair_ok.values()):
    print("🚨 at least one run did not start from the same initial weights / RNG stream as its reused v18 control "
          "(excluding raw_beta): the paired comparison is not clean for that seed.")
if not all(data_checks.values()):
    print("🚨 a data / RNG diagnostic is a MISMATCH — the data or model construction differs from v18 beyond the "
          "intended beta change; the reused control may not be a valid baseline.")

FINAL VALIDATION
  strongest-64 bin indices                                        : match
  train windows / sensor values / timestamps / class counts       : match
  val windows / sensor values / timestamps / class counts         : match
  test windows / sensor values / timestamps / class counts        : match
  seed-42 torch RNG after construction == v16/v18 (no extra draws): match
  fusion backbone is real mamba_ssm                               : match

  control group                                   : v18 beta=0.0 rows REUSED, not retrained (5 seeds)
  each run starts where its reused control started: ✅ all seeds
  zero-init g final layer before training         : ✅ asserted for all 5 runs
  beta(0) / d(beta)/d(raw) at init                : 42: 0.00500 / 0.00499, 123: 0.00500 / 0.00499, 2024: 0.00500 / 0.00499, 7: 0.00500 / 0.00499, 99: 0.00500 / 0.00499  (all > 0.0001: ✅)
  beta never negative (min over every step)       : 2.561e-03 ✅
  LayerNorm on g(Δτ) unchanged from v18    